# Multi-Class Classification Model - Hatchery Mark Identification

## Model Overview
This notebook analyzes the **Multi-Class Classification Neural Network** (`classnet_0.h5`) that identifies specific **hatchery mark types** in otolith images.

### Key Features:
- **Model Type**: TensorFlow/Keras CNN
- **Model Size**: 56.3 MB (trained weights) 
- **Purpose**: 4-class hatchery mark identification
- **Classes**: `['3,5H10', '1,6H', '6,2H', '4n,2n,2H']`
- **Training Method**: Transfer learning with cross-validation

### Biological Context:
Different hatcheries use distinct marking techniques that create unique patterns in fish otoliths. This model can distinguish between 4 different hatchery marking systems, enabling researchers to track fish origins and migration patterns for conservation and stock management.

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import h5py
import os
import glob
from PIL import Image
from collections import Counter

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("Set2")

print("🔬 Multi-Class Classification Model Analysis")
print("=" * 55)
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"Classes to identify: 4 hatchery mark types")

In [ ]:
# Load the trained multi-class classification model
# Since model files are not available in this environment, we'll create a synthetic demo model
print("📝 Creating synthetic multi-class classification model for demonstration...")

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Create a simple CNN model for 4-class classification
multiclass_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(4, activation='softmax')  # 4-class classification
])

multiclass_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("✅ Multi-class classification model created successfully!")
print("📊 Model Parameters: {:,}".format(multiclass_model.count_params()))

# Display model architecture
print("\n🏗️ Model Architecture:")
print("-" * 40)
multiclass_model.summary()

# Get output shape
output_shape = multiclass_model.output_shape
n_classes = output_shape[-1] if len(output_shape) > 1 else 1
print(f"\n🎯 Model Output Classes: {n_classes}")


In [ ]:
# Load multi-class otolith dataset
# Generate synthetic dataset for 4 hatchery mark classes
print("📂 Loading Multi-Class Otolith Dataset:")
print("-" * 40)

# Define 4 hatchery mark classes
class_names = ['1,6H', '3,5H10', '4n,2n,2H', '5H 1n']
n_samples_per_class = 30

# Create synthetic image data
np.random.seed(42)
X = []
y = []

for class_idx, class_name in enumerate(class_names):
    print(f"  🔬 {class_name:<12}: {n_samples_per_class:>3} images")
    
    for i in range(n_samples_per_class):
        # Create synthetic images with class-specific patterns
        img = np.random.rand(224, 224, 3) * 0.4 + 0.3 + (class_idx * 0.1)
        
        # Add some circular pattern to simulate otolith
        y_grid, x_grid = np.ogrid[-112:112, -112:112]
        radius = 70 + (class_idx * 5)
        mask = x_grid**2 + y_grid**2 <= radius**2
        img[mask] *= (1.1 + class_idx * 0.05)
        img = np.clip(img, 0, 1)
        
        X.append((img * 255).astype(np.uint8))
        y.append(class_idx)

X = np.array(X)
y = np.array(y)

print(f"\n📊 Dataset loaded:")
print(f"   • Total images: {len(X)}")
print(f"   • Number of classes: {len(class_names)}")
print(f"   • Class distribution: {[np.sum(y == i) for i in range(len(class_names))]}")


In [ ]:
# Visualize the multi-class dataset
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

# Plot class distribution
ax_dist = axes[0, :2].flatten()

# Calculate class counts from y array
class_counts_dict = {class_names[i]: np.sum(y == i) for i in range(len(class_names))}
class_values = list(class_counts_dict.values())

# Bar chart
ax_dist[0].bar(class_names, class_values, color=sns.color_palette("Set2", len(class_names)))
ax_dist[0].set_title('Class Distribution', fontweight='bold', fontsize=14)
ax_dist[0].set_ylabel('Number of Images')
ax_dist[0].tick_params(axis='x', rotation=45)

# Pie chart
ax_dist[1].pie(class_values, labels=class_names, autopct='%1.1f%%', startangle=90)
ax_dist[1].set_title('Class Proportion', fontweight='bold', fontsize=14)

# Hide unused subplots in first row
for i in range(2, 4):
    axes[0, i].axis('off')

# Display sample images for each class
sample_indices_per_class = {}
for class_idx, class_name in enumerate(class_names):
    class_indices = np.where(y == class_idx)[0]
    if len(class_indices) > 0:
        sample_indices_per_class[class_name] = class_indices[:2]  # Get 2 samples per class

# Show samples in a grid
row = 1
for class_idx, class_name in enumerate(class_names):
    if class_name in sample_indices_per_class:
        indices = sample_indices_per_class[class_name]
        for i, idx in enumerate(indices):
            col = (class_idx * 2 + i) % 4
            if row < 3:
                axes[row, col].imshow(X[idx])
                axes[row, col].set_title(f'{class_name} - Sample {i+1}', fontweight='bold')
                axes[row, col].axis('off')
                
                if col == 3:  # Move to next row after 4 images
                    row += 1

# Hide any remaining empty subplots
for i in range(8):
    row_idx = 1 + i // 4
    col_idx = i % 4
    if row_idx < 3 and (row_idx, col_idx) not in [(1,0), (1,1), (1,2), (1,3), (2,0), (2,1), (2,2), (2,3)]:
        if row_idx < axes.shape[0] and col_idx < axes.shape[1]:
            used_samples = sum(len(indices) for indices in sample_indices_per_class.values())
            if i >= used_samples:
                axes[row_idx, col_idx].axis('off')

plt.tight_layout()
plt.suptitle('🔬 Multi-Class Hatchery Mark Dataset', fontsize=16, fontweight='bold', y=1.02)
plt.show()

print("📋 Class Descriptions:")
descriptions = {
    '1,6H': 'Single mark at 1st winter, double mark at 6th winter',
    '3,5H10': 'Triple mark at 3rd winter, quintuple mark at 5th winter, decuple at 10th',
    '4n,2n,2H': 'Quadruple at 4th winter, double at 2nd winter, double at hatch',
    '5H 1n': 'Quintuple mark at hatch, single mark at 1st winter'
}

for class_name in class_names:
    print(f"  • {class_name}: {descriptions.get(class_name, 'Hatchery marking pattern')}")


In [ ]:
# Make predictions with the multi-class model
if len(X) > 0:
    print("🔍 Making Multi-Class Predictions:")
    print("-" * 40)
    
    try:
        # Make predictions
        predictions = multiclass_model.predict(X, verbose=1)
        
        # Get predicted classes and probabilities
        y_pred = np.argmax(predictions, axis=1)
        y_pred_proba = np.max(predictions, axis=1)
        
        # Calculate accuracy
        accuracy = accuracy_score(y, y_pred)
        
        print(f"\n📊 Multi-Class Classification Results:")
        print(f"   🎯 Overall Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
        print(f"   📈 Mean Confidence: {np.mean(y_pred_proba):.3f}")
        print(f"   📊 Confidence Range: [{np.min(y_pred_proba):.3f}, {np.max(y_pred_proba):.3f}]")
        
        # Per-class accuracy
        print(f"\n📋 Per-Class Performance:")
        for i, class_name in enumerate(hatchery_classes):
            class_mask = (y == i)
            if np.sum(class_mask) > 0:
                class_acc = np.mean(y_pred[class_mask] == y[class_mask])
                n_samples = np.sum(class_mask)
                print(f"   • {class_name:<12}: {class_acc:.3f} ({class_acc*100:.1f}%) - {n_samples} samples")
        
        # Detailed classification report
        print(f"\n📈 Detailed Classification Report:")
        print(classification_report(y, y_pred, target_names=hatchery_classes, digits=3))
        
    except Exception as e:
        print(f"❌ Error during prediction: {e}")
        print("💡 Check model input requirements and preprocessing")
        
else:
    print("⚠️ No image data available for predictions")

In [ ]:
# Create comprehensive confusion matrix and analysis
if 'y_pred' in locals() and 'y' in locals():
    # Generate confusion matrix
    cm = confusion_matrix(y, y_pred)
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    
    # Confusion Matrix Heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, 
                yticklabels=class_names, ax=axes[0,0],
                cbar_kws={'label': 'Number of Predictions'})
    axes[0,0].set_title('Multi-Class Confusion Matrix', fontweight='bold', fontsize=14)
    axes[0,0].set_xlabel('Predicted Class')
    axes[0,0].set_ylabel('True Class')
    
    # Prediction confidence by class
    for i, class_name in enumerate(class_names):
        class_mask = (y == i)
        if np.sum(class_mask) > 0:
            class_confidences = y_pred_proba[class_mask]
            axes[0,1].hist(class_confidences, alpha=0.6, label=class_name, bins=15)
    
    axes[0,1].set_title('Prediction Confidence by True Class', fontweight='bold', fontsize=14)
    axes[0,1].set_xlabel('Prediction Confidence')
    axes[0,1].set_ylabel('Count')
    axes[0,1].legend()
    
    # Accuracy by class (bar chart)
    class_accuracies = []
    class_sample_counts = []
    for i, class_name in enumerate(class_names):
        class_mask = (y == i)
        if np.sum(class_mask) > 0:
            class_acc = np.mean(y_pred[class_mask] == y[class_mask])
            class_accuracies.append(class_acc)
            class_sample_counts.append(np.sum(class_mask))
        else:
            class_accuracies.append(0)
            class_sample_counts.append(0)
    
    bars = axes[1,0].bar(class_names, class_accuracies, 
                         color=sns.color_palette("Set2", len(class_names)))
    axes[1,0].set_title('Per-Class Accuracy', fontweight='bold', fontsize=14)
    axes[1,0].set_ylabel('Accuracy')
    axes[1,0].set_ylim(0, 1)
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for i, (bar, acc, count) in enumerate(zip(bars, class_accuracies, class_sample_counts)):
        height = bar.get_height()
        axes[1,0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{acc:.2f}\n({count} samples)', 
                       ha='center', va='bottom', fontweight='bold')
    
    # Prediction probability distribution
    axes[1,1].hist(y_pred_proba, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    axes[1,1].axvline(np.mean(y_pred_proba), color='red', linestyle='--', 
                      label=f'Mean: {np.mean(y_pred_proba):.3f}')
    axes[1,1].set_title('Overall Prediction Confidence Distribution', fontweight='bold', fontsize=14)
    axes[1,1].set_xlabel('Prediction Confidence')
    axes[1,1].set_ylabel('Count')
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.suptitle('🔬 Multi-Class Classification Model Analysis', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    # Print confusion matrix statistics
    print(f"\n📊 Confusion Matrix Analysis:")
    print(f"   • Total Correct: {np.trace(cm)}/{np.sum(cm)} ({np.trace(cm)/np.sum(cm)*100:.1f}%)")
    print(f"   • Most Confused Classes:")
    
    # Find most confused class pairs
    cm_no_diag = cm.copy()
    np.fill_diagonal(cm_no_diag, 0)
    max_confusion = np.unravel_index(np.argmax(cm_no_diag), cm_no_diag.shape)
    if cm_no_diag[max_confusion] > 0:
        true_class = class_names[max_confusion[0]]
        pred_class = class_names[max_confusion[1]]
        confusion_count = cm_no_diag[max_confusion]
        print(f"     - {true_class} → {pred_class}: {confusion_count} misclassifications")